In [1]:
!pip install yfinance hmmlearn pandas numpy scikit-learn matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 3.8 MB/s eta 0:00:00


In [2]:
import os
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)
print("folders created")

folders created


In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

START_DATE = "2010-01-01"
END_DATE   = "2025-12-31"

TICKERS = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "META", "AMZN", "TSM", "AVGO", "ORCL", "ASML",
    "JPM", "BAC", "WFC", "GS", "MS", "BLK", "AXP", "SPGI", "MCO", "ICE",
    "JNJ", "UNH", "LLY", "PFE", "ABBV", "MRK", "TMO", "ABT", "DHR", "BMY",
    "PG", "KO", "PEP", "WMT", "COST", "MCD", "NKE", "SBUX", "TGT", "HD",
    "XOM", "CVX", "CAT", "DE", "HON", "UPS", "BA", "MMM", "GE", "LMT"
]

print("downloading stock prices...")
prices = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)["Close"]
print(f"got {prices.shape[0]} days for {prices.shape[1]} stocks")

print("downloading VIX...")
vix = yf.download("^VIX", start=START_DATE, end=END_DATE, auto_adjust=True)["Close"]
vix.name = "VIX"

print("downloading SPY...")
spy = yf.download("SPY", start=START_DATE, end=END_DATE, auto_adjust=True)["Close"]
spy.name = "SPY"

print("computing returns...")
returns     = prices.pct_change().dropna()
spy_returns = spy.pct_change().dropna()

missing_pct  = returns.isnull().sum() / len(returns)
good_tickers = missing_pct[missing_pct < 0.20].index
returns      = returns[good_tickers].fillna(0)

common_dates = returns.index.intersection(vix.index).intersection(spy_returns.index)
returns      = returns.loc[common_dates]
vix_aligned  = vix.loc[common_dates]
spy_aligned  = spy_returns.loc[common_dates]

print(f"{len(common_dates)} trading days")
print(f"from {common_dates[0].date()} to {common_dates[-1].date()}")

returns.to_csv("data/stock_returns.csv")
vix_aligned.to_csv("data/vix.csv")
spy_aligned.to_csv("data/spy_returns.csv")
prices.loc[common_dates].to_csv("data/stock_prices.csv")

print("all data saved")

downloading stock prices...


[*********************100%***********************]  50 of 50 completed


got 4023 days for 50 stocks
downloading VIX...


[*********************100%***********************]  1 of 1 completed


downloading SPY...


[*********************100%***********************]  1 of 1 completed


computing returns...
3268 trading days
from 2013-01-03 to 2025-12-30
all data saved


In [4]:
import pandas as pd
import numpy as np

print("loading data...")
returns = pd.read_csv("data/stock_returns.csv", index_col=0, parse_dates=True)
prices  = pd.read_csv("data/stock_prices.csv",  index_col=0, parse_dates=True)
print(f"{returns.shape[0]} days, {returns.shape[1]} stocks")

print("momentum...")
momentum = (
    (1 + returns).rolling(252).apply(np.prod, raw=True) /
    (1 + returns).rolling(21).apply(np.prod, raw=True)
) - 1

print("value...")
high_52w = prices.rolling(252).max()
value    = 1 - (prices / high_52w)
value    = value.loc[returns.index]

print("quality...")
rolling_mean = returns.rolling(63).mean()
rolling_std  = returns.rolling(63).std()
quality      = rolling_mean / (rolling_std + 1e-8)

print("size...")
size = prices.rank(axis=1, ascending=True)
size = size.loc[returns.index]

print("low vol...")
vol_63d = returns.rolling(63).std()
low_vol = 1 / (vol_63d + 1e-8)

print("profitability...")
profitability = (1 + returns).rolling(252).apply(np.prod, raw=True) - 1

print("ranking factors...")
def rank_cross_section(df):
    return df.rank(axis=1, pct=True)

momentum_ranked      = rank_cross_section(momentum)
value_ranked         = rank_cross_section(value)
quality_ranked       = rank_cross_section(quality)
size_ranked          = rank_cross_section(size)
low_vol_ranked       = rank_cross_section(low_vol)
profitability_ranked = rank_cross_section(profitability)

print("building composite score...")
composite = (
    momentum_ranked +
    value_ranked +
    quality_ranked +
    size_ranked +
    low_vol_ranked +
    profitability_ranked
) / 6.0

warmup_days = 252
composite   = composite.iloc[warmup_days:]
returns_cut = returns.iloc[warmup_days:]

print(f"{composite.shape[0]} days after warmup")

composite.to_csv("data/composite_factor.csv")
momentum_ranked.iloc[warmup_days:].to_csv("data/factor_momentum.csv")
value_ranked.iloc[warmup_days:].to_csv("data/factor_value.csv")
quality_ranked.iloc[warmup_days:].to_csv("data/factor_quality.csv")
size_ranked.iloc[warmup_days:].to_csv("data/factor_size.csv")
low_vol_ranked.iloc[warmup_days:].to_csv("data/factor_lowvol.csv")
profitability_ranked.iloc[warmup_days:].to_csv("data/factor_profitability.csv")
returns_cut.to_csv("data/returns_cut.csv")

print("done")

loading data...
3268 days, 50 stocks
momentum...
value...
quality...
size...
low vol...
profitability...
ranking factors...
building composite score...
3016 days after warmup
done


In [5]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM
import os

os.makedirs("results", exist_ok=True)

print("loading data...")
spy_returns = pd.read_csv("data/spy_returns.csv", index_col=0, parse_dates=True)
vix         = pd.read_csv("data/vix.csv",         index_col=0, parse_dates=True)

common      = spy_returns.index.intersection(vix.index)
spy_returns = spy_returns.loc[common]
vix         = vix.loc[common]

spy_returns = spy_returns.iloc[252:]
vix         = vix.iloc[252:]

print(f"{len(spy_returns)} days loaded")

spy_ret_values = spy_returns.values.flatten()
vix_values     = vix.values.flatten()
realized_vol   = pd.Series(spy_ret_values).rolling(21).std().fillna(0).values

X = np.column_stack([
    spy_ret_values,
    vix_values / 100,
    realized_vol
])

print("fitting HMM - this takes a minute...")
model = GaussianHMM(
    n_components=3,
    covariance_type="full",
    n_iter=1000,
    random_state=42
)
model.fit(X)
print(f"converged: {model.monitor_.converged}")

regimes = model.predict(X)
dates   = spy_returns.index

regime_vix = {}
for r in np.unique(regimes):
    mask          = regimes == r
    regime_vix[r] = vix_values[mask].mean()
    print(f"state {r}: avg VIX={regime_vix[r]:.1f}, avg return={spy_ret_values[mask].mean():.4f}")

sorted_states = sorted(regime_vix, key=regime_vix.get)
bull_state    = sorted_states[0]
bear_state    = sorted_states[1]
crisis_state  = sorted_states[2]

print(f"bull={bull_state}, bear={bear_state}, crisis={crisis_state}")

label_map     = {bull_state: "Bull", bear_state: "Bear", crisis_state: "Crisis"}
regime_labels = pd.Series([label_map[r] for r in regimes], index=dates, name="Regime")

print("\nregime breakdown:")
for label in ["Bull", "Bear", "Crisis"]:
    mask    = regime_labels == label
    n       = mask.sum()
    ret     = spy_ret_values[mask.values].mean() * 252
    avg_vix = vix_values[mask.values].mean()
    pct     = n / len(regime_labels) * 100
    print(f"  {label}: {n} days ({pct:.1f}%) | ann.return={ret:.2%} | avg VIX={avg_vix:.1f}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle("S&P 500 regime detection - Hidden Markov Model", fontsize=13)

cum_ret = (1 + pd.Series(spy_ret_values, index=dates)).cumprod()
ax1.plot(dates, cum_ret, color="#1f77b4", linewidth=1)
ax1.set_ylabel("cumulative return")
ax1.grid(True, alpha=0.3)

colors = {"Bull": "#90EE90", "Bear": "#FFD700", "Crisis": "#FF6B6B"}
prev   = regime_labels.iloc[0]
start  = dates[0]

for i, (date, regime) in enumerate(regime_labels.items()):
    if regime != prev or i == len(regime_labels) - 1:
        ax1.axvspan(start, date, alpha=0.2, color=colors[prev], label=prev)
        ax2.axvspan(start, date, alpha=0.4, color=colors[prev])
        start = date
        prev  = regime

regime_numeric = regime_labels.map({"Bull": 1, "Bear": 0, "Crisis": -1})
ax2.plot(dates, regime_numeric, color="#333333", linewidth=0.5)
ax2.set_yticks([-1, 0, 1])
ax2.set_yticklabels(["Crisis", "Bear", "Bull"])
ax2.set_ylabel("regime")
ax2.grid(True, alpha=0.3)

handles, labels = ax1.get_legend_handles_labels()
by_label        = dict(zip(labels, handles))
ax1.legend(by_label.values(), by_label.keys(), loc="upper left")

plt.tight_layout()
plt.savefig("results/regime_chart.png", dpi=150, bbox_inches="tight")
plt.close()
print("saved results/regime_chart.png")

regime_labels.to_csv("data/regime_labels.csv")

print("\ntransition matrix:")
trans = pd.DataFrame(
    model.transmat_,
    index=[label_map[i] for i in range(3)],
    columns=[label_map[i] for i in range(3)]
)
print(trans.round(3))
trans.to_csv("results/transition_matrix.csv")
print("done")

loading data...
3016 days loaded
fitting HMM - this takes a minute...
converged: True
state 0: avg VIX=19.7, avg return=0.0003
state 1: avg VIX=13.4, avg return=0.0010
state 2: avg VIX=31.7, avg return=-0.0006
bull=1, bear=0, crisis=2

regime breakdown:
  Bull: 1506 days (49.9%) | ann.return=26.04% | avg VIX=13.4
  Bear: 1150 days (38.1%) | ann.return=7.90% | avg VIX=19.7
  Crisis: 360 days (11.9%) | ann.return=-14.52% | avg VIX=31.7
saved results/regime_chart.png

transition matrix:
        Bear   Bull  Crisis
Bear    0.96  0.028   0.012
Bull    0.02  0.979   0.001
Crisis  0.04  0.000   0.960
done


In [6]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

os.makedirs("results", exist_ok=True)

print("loading data...")
composite   = pd.read_csv("data/composite_factor.csv",    index_col=0, parse_dates=True)
momentum    = pd.read_csv("data/factor_momentum.csv",     index_col=0, parse_dates=True)
low_vol     = pd.read_csv("data/factor_lowvol.csv",       index_col=0, parse_dates=True)
quality     = pd.read_csv("data/factor_quality.csv",      index_col=0, parse_dates=True)
returns     = pd.read_csv("data/returns_cut.csv",         index_col=0, parse_dates=True)
spy_returns = pd.read_csv("data/spy_returns.csv",         index_col=0, parse_dates=True)
regimes     = pd.read_csv("data/regime_labels.csv",       index_col=0, parse_dates=True)

regimes.columns = ["Regime"]

common = composite.index\
    .intersection(returns.index)\
    .intersection(regimes.index)\
    .intersection(spy_returns.index)

composite   = composite.loc[common]
momentum    = momentum.loc[common]
low_vol     = low_vol.loc[common]
quality     = quality.loc[common]
returns     = returns.loc[common]
spy_returns = spy_returns.loc[common].squeeze()
regimes     = regimes.loc[common]

print(f"{len(common)} days")

FACTOR_MAP = {
    "Bull":   momentum,
    "Bear":   low_vol,
    "Crisis": low_vol
}

EXPOSURE_MAP = {
    "Bull":   1.0,
    "Bear":   1.0,
    "Crisis": 0.5
}

N_STOCKS = 10

print("running monthly rebalance...")

portfolio_returns = []
portfolio_dates   = []
spy_ret_list      = []
holdings_log      = []

month_ends = composite.resample("ME").last().index

for i in range(len(month_ends) - 1):
    rebalance_date = month_ends[i]
    next_rebalance = month_ends[i + 1]

    if rebalance_date not in composite.index:
        continue

    regime   = regimes.loc[rebalance_date, "Regime"]
    factor   = FACTOR_MAP[regime]
    exposure = EXPOSURE_MAP[regime]

    if rebalance_date not in factor.index:
        continue

    factor_scores = factor.loc[rebalance_date].dropna()
    top_stocks    = factor_scores.nlargest(N_STOCKS).index.tolist()

    mask           = (returns.index > rebalance_date) & (returns.index <= next_rebalance)
    period_returns = returns.loc[mask, top_stocks]

    if period_returns.empty:
        continue

    daily_ret = period_returns.mean(axis=1) * exposure

    for date, ret in daily_ret.items():
        portfolio_returns.append(ret)
        portfolio_dates.append(date)
        spy_ret_list.append(spy_returns.loc[date] if date in spy_returns.index else np.nan)

    holdings_log.append({
        "date":     rebalance_date,
        "regime":   regime,
        "stocks":   ", ".join(top_stocks),
        "exposure": exposure
    })

strategy_series = pd.Series(portfolio_returns, index=portfolio_dates).sort_index()
spy_series      = pd.Series(spy_ret_list,      index=portfolio_dates).sort_index()

strategy_series = strategy_series.dropna()
spy_series      = spy_series.dropna()
common_idx      = strategy_series.index.intersection(spy_series.index)
strategy_series = strategy_series.loc[common_idx]
spy_series      = spy_series.loc[common_idx]

def compute_metrics(r, label):
    ann_ret = r.mean() * 252
    ann_vol = r.std()  * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0
    cum     = (1 + r).cumprod()
    max_dd  = ((cum - cum.cummax()) / cum.cummax()).min()
    return {
        "label":           label,
        "ann. return":     f"{ann_ret:.2%}",
        "ann. volatility": f"{ann_vol:.2%}",
        "sharpe":          f"{sharpe:.2f}",
        "max drawdown":    f"{max_dd:.2%}",
        "days":            len(r)
    }

strat_m = compute_metrics(strategy_series, "regime strategy")
spy_m   = compute_metrics(spy_series,      "SPY")

print("\nresults:")
for k, v in strat_m.items():
    print(f"  {k}: {v}")
print()
for k, v in spy_m.items():
    print(f"  {k}: {v}")

cum_strat = (1 + strategy_series).cumprod()
cum_spy   = (1 + spy_series).cumprod()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(cum_strat.index, cum_strat.values, label="regime strategy", linewidth=1.5)
ax.plot(cum_spy.index,   cum_spy.values,   label="SPY",             linewidth=1.5)
ax.set_title("regime strategy vs SPY")
ax.set_ylabel("cumulative return")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("results/strategy_performance.png", dpi=150, bbox_inches="tight")
plt.close()

pd.DataFrame(holdings_log).to_csv("results/holdings_log.csv", index=False)
pd.DataFrame([strat_m, spy_m]).to_csv("results/performance_metrics.csv", index=False)

print("\nsaved results")
print("done")

loading data...
3016 days
running monthly rebalance...

results:
  label: regime strategy
  ann. return: 17.65%
  ann. volatility: 14.22%
  sharpe: 1.24
  max drawdown: -18.55%
  days: 2104

  label: SPY
  ann. return: 15.08%
  ann. volatility: 16.49%
  sharpe: 0.91
  max drawdown: -23.18%
  days: 2104

saved results
done


In [7]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

os.makedirs("results", exist_ok=True)

composite   = pd.read_csv("data/composite_factor.csv",    index_col=0, parse_dates=True)
momentum    = pd.read_csv("data/factor_momentum.csv",     index_col=0, parse_dates=True)
low_vol     = pd.read_csv("data/factor_lowvol.csv",       index_col=0, parse_dates=True)
returns     = pd.read_csv("data/returns_cut.csv",         index_col=0, parse_dates=True)
spy_returns = pd.read_csv("data/spy_returns.csv",         index_col=0, parse_dates=True)
regimes     = pd.read_csv("data/regime_labels.csv",       index_col=0, parse_dates=True)

regimes.columns = ["Regime"]

common = composite.index\
    .intersection(returns.index)\
    .intersection(regimes.index)\
    .intersection(spy_returns.index)

composite   = composite.loc[common]
momentum    = momentum.loc[common]
low_vol     = low_vol.loc[common]
returns     = returns.loc[common]
spy_returns = spy_returns.loc[common].squeeze()
regimes     = regimes.loc[common]

def run_strategy(comp, mom, lv, ret, spy, reg, n_stocks=10, freq="ME"):
    factor_map   = {"Bull": mom, "Bear": lv, "Crisis": lv}
    exposure_map = {"Bull": 1.0, "Bear": 1.0, "Crisis": 0.5}
    port, spy_r, dates = [], [], []
    ends = comp.resample(freq).last().index
    for i in range(len(ends) - 1):
        rd = ends[i]
        nd = ends[i + 1]
        if rd not in comp.index or rd not in reg.index:
            continue
        regime   = reg.loc[rd, "Regime"]
        factor   = factor_map[regime]
        exposure = exposure_map[regime]
        if rd not in factor.index:
            continue
        scores = factor.loc[rd].dropna()
        top    = scores.nlargest(n_stocks).index.tolist()
        mask   = (ret.index > rd) & (ret.index <= nd)
        period = ret.loc[mask, top]
        if period.empty:
            continue
        for date, r in (period.mean(axis=1) * exposure).items():
            port.append(r)
            dates.append(date)
            spy_r.append(spy.loc[date] if date in spy.index else np.nan)
    s   = pd.Series(port,  index=dates).dropna()
    b   = pd.Series(spy_r, index=dates).dropna()
    idx = s.index.intersection(b.index)
    return s.loc[idx], b.loc[idx]

def sharpe(r):
    return (r.mean() * 252) / (r.std() * np.sqrt(252)) if r.std() > 0 else 0

print("test 1: varying number of stocks...")
results_stocks = []
for n in [5, 10, 20, 30]:
    s, b = run_strategy(composite, momentum, low_vol, returns, spy_returns, regimes, n_stocks=n)
    results_stocks.append({
        "n stocks":        n,
        "strategy sharpe": round(sharpe(s), 2),
        "spy sharpe":      round(sharpe(b), 2),
        "strategy return": f"{s.mean()*252:.2%}"
    })
    print(f"  n={n}: strategy={sharpe(s):.2f}, spy={sharpe(b):.2f}")

pd.DataFrame(results_stocks).to_csv("results/robustness_n_stocks.csv", index=False)

print("\ntest 2: rebalancing frequency...")
results_freq = []
for freq, label in [("ME", "monthly"), ("QE", "quarterly")]:
    s, b = run_strategy(composite, momentum, low_vol, returns, spy_returns, regimes, freq=freq)
    results_freq.append({
        "frequency":       label,
        "strategy sharpe": round(sharpe(s), 2),
        "spy sharpe":      round(sharpe(b), 2),
        "strategy return": f"{s.mean()*252:.2%}"
    })
    print(f"  {label}: strategy={sharpe(s):.2f}, spy={sharpe(b):.2f}")

pd.DataFrame(results_freq).to_csv("results/robustness_frequency.csv", index=False)

print("\ntest 3: in-sample vs out-of-sample (split at 2019)...")
split = "2019-01-01"
for label, before_split in [("in-sample (2010-2018)", True), ("out-of-sample (2019-2025)", False)]:
    mask = composite.index < split if before_split else composite.index >= split
    s, b = run_strategy(
        composite.loc[mask], momentum.loc[mask], low_vol.loc[mask],
        returns.loc[mask],   spy_returns.loc[mask], regimes.loc[mask]
    )
    if len(s) < 50:
        continue
    print(f"\n  {label}")
    print(f"    strategy sharpe: {sharpe(s):.2f}")
    print(f"    spy sharpe:      {sharpe(b):.2f}")
    print(f"    strategy return: {s.mean()*252:.2%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("robustness tests", fontsize=12)

axes[0].bar(
    [str(r["n stocks"]) for r in results_stocks],
    [r["strategy sharpe"] for r in results_stocks],
    color="#1f77b4", alpha=0.7
)
axes[0].axhline(results_stocks[0]["spy sharpe"], color="orange", linestyle="--", label="SPY")
axes[0].set_title("sharpe vs n stocks")
axes[0].set_xlabel("stocks held")
axes[0].set_ylabel("sharpe ratio")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(
    [r["frequency"] for r in results_freq],
    [r["strategy sharpe"] for r in results_freq],
    color="#2ca02c", alpha=0.7
)
axes[1].set_title("sharpe vs rebalance frequency")
axes[1].set_ylabel("sharpe ratio")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("results/robustness_summary.png", dpi=150, bbox_inches="tight")
plt.close()

print("\nsaved results/robustness_summary.png")
print("done")

test 1: varying number of stocks...
  n=5: strategy=1.19, spy=0.91
  n=10: strategy=1.24, spy=0.91
  n=20: strategy=1.12, spy=0.91
  n=30: strategy=1.15, spy=0.91

test 2: rebalancing frequency...
  monthly: strategy=1.24, spy=0.91
  quarterly: strategy=0.65, spy=0.72

test 3: in-sample vs out-of-sample (split at 2019)...

  in-sample (2010-2018)
    strategy sharpe: 1.13
    spy sharpe:      0.73
    strategy return: 17.74%

  out-of-sample (2019-2025)
    strategy sharpe: 1.33
    spy sharpe:      0.95
    strategy return: 17.59%

saved results/robustness_summary.png
done


In [8]:
from google.colab import files
import os

print("downloading results...")
for f in os.listdir("results"):
    files.download(f"results/{f}")
    print(f"downloaded: {f}")
print("done")

downloading results...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded: performance_metrics.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

downloaded: robustness_n_stocks.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded: strategy_performance.png
done
